In [ ]:
!pip install -q executorch
import executorch, torch, torchvision, os
from importlib.metadata import version
print("executorch:", version("executorch"))
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pth", ".json")):
            print(os.path.join(root, f))

In [1]:
!pip install -q torchvision "torch==2.15.0"
import executorch, torch, torchvision, os
from importlib.metadata import version
print("executorch:", version("executorch"))
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pth", ".json")):
            print(os.path.join(root, f))

executorch: 1.6.0
torch: 2.15.0+cu132 | torchvision: 0.29.2+cu132
/kaggle/input/datasets/viroshans/crop-model-v3/best_model_v3.pth
/kaggle/input/datasets/viroshans/crop-model-v3/classes_v3.json


In [2]:
import json, torch, torch.nn as nn
from torchvision import models
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
from executorch.runtime import Runtime

D = "/kaggle/input/datasets/viroshans/crop-model-v3"
CL = json.load(open(f"{D}/classes_v3.json"))
m = models.mobilenet_v3_small()
m.classifier[3] = nn.Linear(m.classifier[3].in_features, len(CL))
m.load_state_dict(torch.load(f"{D}/best_model_v3.pth", map_location="cpu"))
m.eval()

x = torch.randn(1, 3, 224, 224)
ep = torch.export.export(m, (x,))
et = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch()
with open("/kaggle/working/vision.pte", "wb") as f:
    f.write(et.buffer)
print("saved vision.pte, size MB:", round(len(et.buffer) / 1e6, 2), "| classes:", len(CL))

# verify: the .pte output must match the PyTorch model
prog = Runtime.get().load_program("/kaggle/working/vision.pte")
out = prog.load_method("forward").execute([x])[0]
with torch.no_grad():
    ref = m(x)
print("max abs difference:", (out - ref).abs().max().item())
print("same top class:", out.argmax().item() == ref.argmax().item())

W1113 15:54:31.682000 148 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


saved vision.pte, size MB: 6.16 | classes: 13
max abs difference: 1.2636184692382812e-05
same top class: True
